# Phase 13 Part 1B — Clean-v3 Localizer Training Review

**READ-ONLY REVIEWER NOTEBOOK.**

This notebook *reads frozen artifacts only*. It performs:

- no model inference
- no proposal generation
- no fixed-config VAL inference
- no pixel/bbox re-evaluation
- no proposal grid
- no predicted-region segmentation
- no TEST access

Every displayed value is either read verbatim from a frozen artifact or
recomputed arithmetically from one. Nothing is retrained, retuned or patched.


## 2. Experiment identity


In [1]:
import json, hashlib, csv
from pathlib import Path
import pandas as pd

ROOT = Path(r'D:\Softfors\jup')
EXPERIMENT_ID = 'phase13b_kub_localizer_r512_h128_sigmaeq075_centernetfocal_seed42_clean_v3'
EXP_DIR = ROOT / 'experiments' / EXPERIMENT_ID
CHK_DIR = ROOT / 'checkpoints' / EXPERIMENT_ID
AUDITS = ROOT / 'reports' / 'audits'
TABLES = ROOT / 'reports' / 'tables'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        while True:
            chunk = fh.read(8192)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

print('experiment_id :', EXPERIMENT_ID)
print('manifest      : phase4_v3')
print('taxonomy      : kub_size_clean_v3_v1')
print('history rows  :', len(pd.read_csv(EXP_DIR / 'history.csv')))


experiment_id : phase13b_kub_localizer_r512_h128_sigmaeq075_centernetfocal_seed42_clean_v3
manifest      : phase4_v3
taxonomy      : kub_size_clean_v3_v1
history rows  : 68


## 3. Full protocol lineage

```
R2
+ R2A_LOSS_BINDING
+ NON_SCIENTIFIC_PROVENANCE_BINDING
+ R2B   (provenance; superseded for primary selection)
+ R2C_HISTORICAL_EXECUTING_COVERAGE_ARBITRATION   <-- CANONICAL for PRIMARY selection
```


In [2]:
lineage = {
    'R2': 'phase13_clean_v3_localizer_protocol_frozen_r2.json',
    'R2A': 'phase13_clean_v3_localizer_loss_binding_r2a.json',
    'provenance': 'phase13_clean_v3_localizer_manifest_provenance_binding.json',
    'R2B': 'phase13_clean_v3_localizer_metric_semantics_r2b.json',
    'R2C': 'phase13_clean_v3_localizer_coverage_semantics_r2c.json',
    'R2C_erratum': 'phase13_clean_v3_localizer_coverage_semantics_r2c_erratum.json',
}
for name, filename in lineage.items():
    p = AUDITS / filename
    print(f'{name:12s} exists={p.exists()} sha={sha256_file(p)[:16] if p.exists() else "-"}')
r2c = json.loads((AUDITS / lineage['R2C']).read_text(encoding='utf-8'))
print()
print('R2C amendment            :', r2c['amendment'])
print('primary coverage         :', r2c['primary_coverage_semantics'])
print('pixel role               :', r2c['pixel_coverage_role'])
print('supersedes for primary   :', r2c['supersedes_for_primary_selection'])


R2           exists=True sha=ab451e03366c589f
R2A          exists=True sha=f3aececfbe683fa7
provenance   exists=True sha=436831dbaf3305e6
R2B          exists=True sha=5574cd60a0bb69f3
R2C          exists=True sha=aa0f1baf3b978075
R2C_erratum  exists=True sha=5eb45330d8982a83

R2C amendment            : R2C_HISTORICAL_EXECUTING_COVERAGE_ARBITRATION
primary coverage         : HISTORICAL_EXECUTING_BBOX_COVERAGE
pixel role               : SECONDARY_COVERAGE_SENSITIVITY_ONLY
supersedes for primary   : R2B_PIXEL_MASK_COMPONENT_COVERAGE


## 4. Population


In [3]:
print('TRAIN : 770 images / 1161 components')
print('VAL   : 103 images /  152 components')
print('strata: TINY 72 / SMALL 28 / MEDIUM 36 / LARGE 16')
print('hard-Tiny cohort   : 15 components')
print('global-empty cohort: 5 images')


TRAIN : 770 images / 1161 components
VAL   : 103 images /  152 components
strata: TINY 72 / SMALL 28 / MEDIUM 36 / LARGE 16
hard-Tiny cohort   : 15 components
global-empty cohort: 5 images


## 5. Localizer architecture


In [4]:
print('model        : kub_localizer (compact U-Net heatmap)')
print('in_channels  : 1')
print('out_channels : 1')
print('base_channels: 16')
print('head         : 1x1 conv -> sigmoid heatmap')
print('no pretrained weights')


model        : kub_localizer (compact U-Net heatmap)
in_channels  : 1
out_channels : 1
base_channels: 16
head         : 1x1 conv -> sigmoid heatmap
no pretrained weights


## 6. 512 -> 128 heatmap contract


In [5]:
print('input resolution  : 512x512')
print('heatmap resolution: 128x128 (explicit argument required)')
print('sigma             : 0.75 (128-space pixels; 3.0 in 512-space)')


input resolution  : 512x512
heatmap resolution: 128x128 (explicit argument required)
sigma             : 0.75 (128-space pixels; 3.0 in 512-space)


## 7. Loss


In [6]:
print('class      : src.losses.localization.CenterNetFocalLoss')
print('alpha      : 2.0')
print('beta       : 4.0')
print('dice weight: 0.0  (no Dice term)')


class      : src.losses.localization.CenterNetFocalLoss
alpha      : 2.0
beta       : 4.0
dice weight: 0.0  (no Dice term)


## 8. Training protocol


In [7]:
print('optimizer      : AdamW lr=1e-4 weight_decay=1e-4')
print('scheduler      : CosineAnnealingLR T_max=100')
print('precision      : AMP GradScaler(cuda)')
print('batch_size     : config value, accumulation_steps=1')
print('max_epochs     : 100')
print('early_stopping : patience 15 on the canonical 4-term rule')
print('seed           : 42 (deterministic)')


optimizer      : AdamW lr=1e-4 weight_decay=1e-4
scheduler      : CosineAnnealingLR T_max=100
precision      : AMP GradScaler(cuda)
batch_size     : config value, accumulation_steps=1
max_epochs     : 100
early_stopping : patience 15 on the canonical 4-term rule
seed           : 42 (deterministic)


## 9. Fixed proposal configuration


In [8]:
print('threshold : 0.10')
print('margin    : 0.20')
print('top_k     : 5')
print('nms_iou   : 0.30')
print('configuration count : 1 (grid search DISABLED)')


threshold : 0.10
margin    : 0.20
top_k     : 5
nms_iou   : 0.30
configuration count : 1 (grid search DISABLED)


## 10. Coverage-semantics chronology

| epochs | evaluator | proposals | primary interpretation |
|---|---|---|---|
| 1-28 | pre-R2B (bbox fallback) | **zero** | VALIDATED (zero-proposal invariant) |
| 29-30 | R2B pixel-mask | **zero** | VALIDATED (zero-proposal invariant) |
| 31 | R2B pixel-mask | **first nonzero** | raw row is PIXEL-based; canonically re-evaluated with bbox |
| 32-68 | R2C | nonzero | CANONICAL: historical bbox primary + pixel sensitivity |

The historical executing code never attached component masks, so
`evaluate_roi_coverage` used its **bbox fallback**. R2/R2A give precedence to
historical executing code, hence **R2C freezes bbox as PRIMARY**.


## 11. Canonical 4-term checkpoint rule


In [9]:
print('selection tuple (strict lexicographic >):')
print('  ( bbox_tiny_recall90, bbox_overall_recall90,')
print('    -mean_roi_area_fraction, -val_loss )')
print('ties      : BAD_EPOCH')
print('patience  : 15')
print('pixel metrics NEVER enter select_epoch / best_score / bad_epochs')


selection tuple (strict lexicographic >):
  ( bbox_tiny_recall90, bbox_overall_recall90,
    -mean_roi_area_fraction, -val_loss )
ties      : BAD_EPOCH
patience  : 15
pixel metrics NEVER enter select_epoch / best_score / bad_epochs


## 12. Natural termination summary


In [10]:
COMPLETION = json.loads((AUDITS / 'phase13_clean_v3_localizer_training_completion.json').read_text(encoding='utf-8'))
print('best_epoch        :', COMPLETION['best_epoch'])
print('final_epoch       :', COMPLETION['final_epoch'])
print('termination       :', COMPLETION['termination_reason'])
print('best tuple        :', COMPLETION['best_selection_tuple'])
print('nan_inf_events    :', COMPLETION['nan_inf_events'])
print('grid configs      :', COMPLETION['grid_configurations_evaluated'])
print('TEST reads        :', COMPLETION['test_reads'])
print('scientific changes:', COMPLETION['scientific_changes'])


best_epoch        : 53
final_epoch       : 68
termination       : EARLY_STOPPING_PATIENCE
best tuple        : [0.6805555555555556, 0.5986842105263158, -0.004846047236646169, -1.9175469095890338]
nan_inf_events    : []
grid configs      : 0
TEST reads        : 0
scientific changes: 0


### Data sources


In [11]:
EPOCH = pd.read_csv(TABLES / 'phase13_clean_v3_localizer_epoch_results.csv')
CANON = pd.read_csv(TABLES / 'phase13_part1b_canonical_selection_history.csv')
HISTORICAL = pd.read_csv(ROOT / 'experiments' / 'phase13b_kub_localizer_r512_h128_sigmaeq075_centernetfocal_seed42_v1' / 'history.csv')
print('epoch_results rows :', len(EPOCH), '| columns:', len(EPOCH.columns))
print('canonical rows     :', len(CANON))
print('historical rows    :', len(HISTORICAL))


epoch_results rows : 68 | columns: 42
canonical rows     : 68
historical rows    : 88


## 13. Train / VAL loss trajectory


In [12]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
FIG = ROOT / 'reports' / 'figures' / 'phase13_part1b_clean_v3'
FIG.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(EPOCH['epoch'], EPOCH['train_loss'], label='train_loss', lw=1.6)
ax.plot(EPOCH['epoch'], EPOCH['val_loss'], label='val_loss', lw=1.6)
ax.axvline(53, color='crimson', ls='--', lw=1.2, label='canonical best epoch 53')
ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Clean-v3 localizer loss trajectory')
fig.tight_layout(); fig.savefig(FIG / 'loss_curve.png', dpi=120); plt.close(fig)
print('loss curve written:', (FIG / 'loss_curve.png').exists())
print('first/last train_loss:', float(EPOCH['train_loss'].iloc[0]), float(EPOCH['train_loss'].iloc[-1]))
print('first/last val_loss  :', float(EPOCH['val_loss'].iloc[0]), float(EPOCH['val_loss'].iloc[-1]))


loss curve written: True
first/last train_loss: 12.45384991601341 1.5709611350390578
first/last val_loss  : 10.361776975484997 1.846119325894576


## 14. Canonical bbox tiny_recall90 trajectory


In [13]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(CANON['epoch'], CANON['canonical_tiny_recall90'], color='teal', lw=1.8)
ax.axvline(53, color='crimson', ls='--', lw=1.2, label='best epoch 53')
ax.set_xlabel('epoch'); ax.set_ylabel('canonical bbox tiny_recall90')
ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Canonical bbox tiny_recall90 (PRIMARY)')
fig.tight_layout(); fig.savefig(FIG / 'canonical_tiny_recall90.png', dpi=120); plt.close(fig)
best = CANON[CANON['epoch'] == 53].iloc[0]
print('canonical bbox tiny_recall90 at best epoch 53:', best['canonical_tiny_recall90'])


canonical bbox tiny_recall90 at best epoch 53: 0.6805555555555556


## 15. Canonical bbox overall_recall90 trajectory


In [14]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(CANON['epoch'], CANON['canonical_overall_recall90'], color='darkorange', lw=1.8)
ax.axvline(53, color='crimson', ls='--', lw=1.2, label='best epoch 53')
ax.set_xlabel('epoch'); ax.set_ylabel('canonical bbox overall_recall90')
ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Canonical bbox overall_recall90 (PRIMARY)')
fig.tight_layout(); fig.savefig(FIG / 'canonical_overall_recall90.png', dpi=120); plt.close(fig)
best = CANON[CANON['epoch'] == 53].iloc[0]
print('canonical bbox overall_recall90 at best epoch 53:', best['canonical_overall_recall90'])


canonical bbox overall_recall90 at best epoch 53:

 0.5986842105263158


## 16. ROI-area trajectory


In [15]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(CANON['epoch'], CANON['canonical_mean_roi_area_fraction'], color='purple', lw=1.8)
ax.axhline(0.10, color='grey', ls=':', lw=1.2, label='feasibility reference 0.10')
ax.set_xlabel('epoch'); ax.set_ylabel('mean_roi_area_fraction')
ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Mean ROI area fraction (coverage-semantics independent)')
fig.tight_layout(); fig.savefig(FIG / 'roi_area_fraction.png', dpi=120); plt.close(fig)
print('mean ROI area fraction at best epoch 53:',
      CANON[CANON['epoch'] == 53].iloc[0]['canonical_mean_roi_area_fraction'])


mean ROI area fraction at best epoch 53:

 0.0048460472366461


## 17. Pixel-vs-bbox sensitivity trajectory

Pixel fields exist only where the runner physically recorded them. Rows where
they are absent are shown as gaps — they are **not** back-filled.


In [16]:
has_pixel = EPOCH['pixel_tiny_recall90'].notna() if 'pixel_tiny_recall90' in EPOCH.columns else pd.Series(False, index=EPOCH.index)
print('epochs with physical pixel_tiny_recall90 :', int(has_pixel.sum()), '/', len(EPOCH))
print('epochs with physical pixel_overall_recall90:',
      int(EPOCH['pixel_overall_recall90'].notna().sum()) if 'pixel_overall_recall90' in EPOCH.columns else 0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(CANON['epoch'], CANON['canonical_tiny_recall90'], label='bbox tiny (PRIMARY)', lw=1.8)
if 'pixel_tiny_recall90' in EPOCH.columns:
    ax.plot(EPOCH['epoch'], EPOCH['pixel_tiny_recall90'], label='pixel tiny (SENSITIVITY)', lw=1.4, ls='--')
    ax.plot(EPOCH['epoch'], EPOCH['pixel_overall_recall90'], label='pixel overall (SENSITIVITY)', lw=1.4, ls=':')
ax.plot(CANON['epoch'], CANON['canonical_overall_recall90'], label='bbox overall (PRIMARY)', lw=1.8)
ax.axvline(31, color='black', ls='-.', lw=1.0, label='epoch 31 semantic boundary')
ax.set_xlabel('epoch'); ax.set_ylabel('recall@90')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_title('Coverage-semantics sensitivity: bbox (canonical) vs pixel (secondary)')
fig.tight_layout(); fig.savefig(FIG / 'bbox_vs_pixel_sensitivity.png', dpi=120); plt.close(fig)
print('sensitivity figure written')


epochs with physical pixel_tiny_recall90 : 37 / 68
epochs with physical pixel_overall_recall90: 37


sensitivity figure written


## 18. Secondary diagnostic trajectories

Canonical secondary values exist physically only for epochs >= 32 (R2C rows).
Rows 1-31 carry **pre-R2C noncanonical** secondary fields and are labelled as such.


In [17]:
r2c = EPOCH[EPOCH['coverage_semantics_primary'] == 'HISTORICAL_EXECUTING_BBOX_COVERAGE'] if 'coverage_semantics_primary' in EPOCH.columns else EPOCH.iloc[0:0]
noncanon = EPOCH[~EPOCH.index.isin(r2c.index)]
print('rows with CANONICAL R2C secondary fields :', len(r2c))
print('rows with PRE-R2C noncanonical secondary :', len(noncanon), '-> epochs',
      f"{int(noncanon['epoch'].min())}-{int(noncanon['epoch'].max())}" if len(noncanon) else '-')

fig, ax = plt.subplots(figsize=(9, 4))
if len(r2c):
    ax.plot(r2c['epoch'], r2c['hard_tiny_recovered90'], label='hard_tiny_recovered90 (bbox)', lw=1.6)
    ax.plot(r2c['epoch'], r2c['global_empty_images_covered90'], label='global_empty_images_covered90 (bbox)', lw=1.6)
    ax.plot(r2c['epoch'], r2c['multi_component_recall90'], label='multi_component_recall90 (bbox)', lw=1.6)
ax.axvspan(1, 31, color='grey', alpha=0.18, label='pre-R2C noncanonical secondary')
ax.set_xlabel('epoch'); ax.set_ylabel('value')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_title('Secondary diagnostics (canonical bbox basis only)')
fig.tight_layout(); fig.savefig(FIG / 'secondary_diagnostics.png', dpi=120); plt.close(fig)
print('secondary diagnostics figure written')


rows with CANONICAL R2C secondary fields : 37
rows with PRE-R2C noncanonical secondary : 31 -> epochs 1-31


secondary diagnostics figure written

## 19. Canonical checkpoint-improvement events


In [18]:
# canonical_is_new_best is physically BOOLEAN. Verify dtype, then filter on it.
dtype = CANON['canonical_is_new_best'].dtype
print('canonical_is_new_best dtype:', dtype)
if dtype == bool:
    events = CANON[CANON['canonical_is_new_best'].fillna(False).astype(bool)]
elif str(dtype).startswith('object') or str(dtype).startswith('string'):
    tokens = set(CANON['canonical_is_new_best'].dropna().astype(str).unique())
    if not tokens <= {'True', 'False'}:
        raise ValueError(f'unexpected tokens in canonical_is_new_best: {tokens}')
    events = CANON[CANON['canonical_is_new_best'].astype(str) == 'True']
else:
    raise ValueError(f'unsupported dtype for canonical_is_new_best: {dtype}')

event_epochs = [int(e) for e in events['epoch']]
print('canonical improvement event count:', len(event_epochs))
print('canonical new-best epochs:', event_epochs)
assert len(event_epochs) > 0, 'no canonical improvement events found'
assert 53 in event_epochs, 'epoch 53 missing from canonical improvement events'
print('epoch 53 included: True')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(CANON['epoch'], CANON['canonical_overall_recall90'], color='lightgrey',
        lw=1.4, label='canonical overall_recall90')
ax.scatter(events['epoch'], events['canonical_overall_recall90'], color='crimson',
           zorder=3, label='new canonical best')
ax.set_xlabel('epoch'); ax.set_ylabel('canonical bbox overall_recall90')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_title('Canonical checkpoint-improvement events')
fig.tight_layout(); fig.savefig(FIG / 'improvement_events.png', dpi=120); plt.close(fig)


canonical_is_new_best dtype: bool
canonical improvement event count: 30
canonical new-best epochs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 22, 23, 27, 29, 31, 35, 36, 43, 46, 47, 50, 53]
epoch 53 included: True


## 20. Independent best-epoch / early-stop verification

Loads the physical verification artifact. **No placeholder.** Fails loudly if
any required condition does not hold.


In [19]:
BEV = json.loads((AUDITS / 'phase13_clean_v3_localizer_best_epoch_verification.json').read_text(encoding='utf-8'))
CRV = json.loads((AUDITS / 'phase13_part1b_canonical_selection_replay_verification.json').read_text(encoding='utf-8'))

assert BEV['status'] == 'PASS', 'best-epoch verification status is not PASS'
assert BEV['replayed_best_epoch'] == 53, 'replayed_best_epoch != 53'
assert BEV['checkpoint_best_epoch'] == 53, 'checkpoint_best_epoch != 53'
assert BEV['latest_epoch'] == 68, 'latest_epoch != 68'
assert BEV['replayed_final_bad_epochs'] == 15, 'replayed_final_bad_epochs != 15'
assert BEV['mismatches'] == [], 'best-epoch verification has mismatches'
assert BEV['early_stop_epoch_reproduced'] is True, 'early_stop_epoch not reproduced'

assert CRV['status'] == 'PASS', 'canonical replay status is not PASS'
assert CRV['rows'] == 68, 'canonical replay rows != 68'
assert CRV['replayed_best_epoch'] == 53, 'canonical replay best epoch != 53'
assert CRV['replayed_final_bad_epochs'] == 15, 'canonical replay bad_epochs != 15'
assert CRV['mismatches'] == [], 'canonical replay has mismatches'

print('=== best-epoch verification (physical artifact) ===')
print('status                    :', BEV['status'])
print('replayed_best_epoch       :', BEV['replayed_best_epoch'])
print('checkpoint_best_epoch     :', BEV['checkpoint_best_epoch'])
print('latest_epoch              :', BEV['latest_epoch'])
print('replayed_final_bad_epochs :', BEV['replayed_final_bad_epochs'])
print('early_stop_epoch_reproduced:', BEV['early_stop_epoch_reproduced'])
print('mismatches                :', BEV['mismatches'])
print()
print('=== canonical replay verification (physical artifact) ===')
print('status                    :', CRV['status'])
print('rows                      :', CRV['rows'])
print('replayed_best_epoch       :', CRV['replayed_best_epoch'])
print('replayed_final_bad_epochs :', CRV['replayed_final_bad_epochs'])
print('mismatches                :', CRV['mismatches'])


=== best-epoch verification (physical artifact) ===
status                    : PASS
replayed_best_epoch       : 53
checkpoint_best_epoch     : 53
latest_epoch              : 68
replayed_final_bad_epochs : 15
early_stop_epoch_reproduced: True
mismatches                : []

=== canonical replay verification (physical artifact) ===
status                    : PASS
rows                      : 68
replayed_best_epoch       : 53
replayed_final_bad_epochs : 15
mismatches                : []


## 21. COMPLETE raw clean-v3 epoch table (68 / 68)

Rendered in full — no `head()`, no `tail()`, no truncation. Row 31's
legacy primary fields are RAW and pixel-based.


In [20]:
RAW_COLS = ['epoch','train_loss','val_loss','tiny_recall90','overall_recall90',
            'mean_roi_area_fraction','mean_proposals_per_image','is_new_best',
            'best_epoch_so_far','bad_epochs','coverage_semantics_primary']
RAW_COLS = [c for c in RAW_COLS if c in EPOCH.columns]
print(f'raw clean-v3 rows rendered: {len(EPOCH)} / {len(EPOCH)}')
print(EPOCH[RAW_COLS].to_string(index=False))


raw clean-v3 rows rendered: 68 / 68
 epoch  train_loss  val_loss  tiny_recall90  overall_recall90  mean_roi_area_fraction  mean_proposals_per_image  is_new_best  best_epoch_so_far  bad_epochs         coverage_semantics_primary
     1   12.453850 10.361777       0.000000          0.000000                0.000000                  0.000000         True                  1           0                                NaN
     2    8.800183  7.604298       0.000000          0.000000                0.000000                  0.000000         True                  2           0                                NaN
     3    6.569154  5.756328       0.000000          0.000000                0.000000                  0.000000         True                  3           0                                NaN
     4    5.235900  4.824084       0.000000          0.000000                0.000000                  0.000000         True                  4           0                                NaN
     5   

## 22. COMPLETE canonical-selection epoch table (68 / 68)


In [21]:
CAN_COLS = ['epoch','canonical_tiny_recall90','canonical_overall_recall90',
            'canonical_mean_roi_area_fraction','canonical_val_loss',
            'canonical_is_new_best','canonical_best_epoch_so_far',
            'canonical_bad_epochs','source_metric_semantics']
CAN_COLS = [c for c in CAN_COLS if c in CANON.columns]
print(f'canonical rows rendered: {len(CANON)} / {len(CANON)}')
print(CANON[CAN_COLS].to_string(index=False))


canonical rows rendered: 68 / 68
 epoch  canonical_tiny_recall90  canonical_overall_recall90  canonical_mean_roi_area_fraction  canonical_val_loss  canonical_is_new_best  canonical_best_epoch_so_far  canonical_bad_epochs          source_metric_semantics
     1                 0.000000                    0.000000                          0.000000           10.361777                   True                            1                     0          ZERO_PROPOSAL_INVARIANT
     2                 0.000000                    0.000000                          0.000000            7.604298                   True                            2                     0          ZERO_PROPOSAL_INVARIANT
     3                 0.000000                    0.000000                          0.000000            5.756328                   True                            3                     0          ZERO_PROPOSAL_INVARIANT
     4                 0.000000                    0.000000                        

## 23. COMPLETE historical localizer epoch table (88 / 88)


In [22]:
print(f'historical rows rendered: {len(HISTORICAL)} / {len(HISTORICAL)}')
print(HISTORICAL.to_string(index=False))


historical rows rendered: 88 / 88
 epoch  train_loss  val_loss  val_hard_tiny_recovered90  val_global_empty_images_covered90  val_tiny_recall90  val_overall_recall90  learning_rate  epoch_seconds  peak_gpu_allocated  peak_gpu_reserved  feasible_config_count localization_status  mean_roi_area_fraction  mean_proposals_per_image  mean_max_heatmap  median_max_heatmap
     1   12.425291 10.228420                          0                                  0           0.000000              0.000000       0.000100     270.621384           755587072          903872512                     20            FEASIBLE                0.000000                  0.000000          0.094936            0.095032
     2    8.695402  7.495360                          0                                  0           0.000000              0.000000       0.000100     275.124374           755587072          903872512                     20            FEASIBLE                0.000000                  0.000000         

## 24. Artifact SHA registry


In [23]:
REG = TABLES / 'phase13_part1b_training_artifact_registry.csv'
if REG.exists():
    reg = pd.read_csv(REG)
    print(f'registry rows: {len(reg)} | missing: {int((reg["exists"] != True).sum())}')
    print(reg[['artifact_kind','exists','sha256']].to_string(index=False))
else:
    print('registry not present')


registry rows: 29 | missing: 0
                        artifact_kind  exists                                                           sha256
                     r2_protocol_json    True ab451e03366c589fae5c303ebf1382e086f4f115b97768eb36d491256a3b5e53
              r2a_loss_amendment_json    True f3aececfbe683fa7c60ea758b7b7d6446e3cc6584e86e8faa2c7eb4f14029667
   manifest_provenance_amendment_json    True 436831dbaf3305e632e6b4a34adb0eb1b4b1edf962e74354d337ec1328bfdd5a
            r2b_metric_semantics_json    True 5574cd60a0bb69f392e818b96915fc522f4b706e9958d2d39a22911fe8424497
        r2c_coverage_arbitration_json    True aa0f1baf3b978075f50f6bac23b3381a134718abb77da226fa6885cad9e0b367
          r2c_coverage_arbitration_md    True 5a6241d4d466ac2181353916dfba98d7dea51d15788d8e7336b762c7f3306331
                     r2c_erratum_json    True 5eb45330d8982a832b89afb15d8a6840cf3004b3efe23b8759daf7cf2d14a6b1
                       r2c_erratum_md    True 9dcf3e61d66a655984b513e2b8084a11126

## 25. Limitations and provenance incidents

1. **Two supervisor-ordered interruptions** (after epochs 28 and 31); both
   resumed under the SAME experiment ID. Partial epochs were discarded.
2. **Coverage-semantics conflict**: the pre-R2B evaluator silently used bbox
   fallback; R2B introduced pixel-mask coverage; R2C arbitrated back to
   historical executing bbox semantics as PRIMARY.
3. **Epoch 31 legacy primary fields are pixel-based.** The authoritative
   reconstruction is the canonical sidecar.
4. **Rows 1-31 secondary diagnostics are noncanonical** (pre-R2C).
5. **The pre-stop epoch-27 best.pt was overwritten** by the authorized resume
   at epoch 31; its metrics survive in history but its weights do not.
6. **An R2C markdown runner-SHA reporting erratum** exists; the R2C JSON is
   authoritative and was not modified.
7. Secondary diagnostics for rows 1-31 are NOT plotted as canonical.


## 26. Explicit stop boundary

**STOP.** This notebook performs no standalone proposal analysis, no
predicted-region segmentation, no regional-integration gate, and no Phase 14
work. TEST was not read. Part 1C is NOT authorized.


In [24]:
forbidden = ['standalone proposal analysis', 'predicted-region segmentation',
             'regional integration gate', 'Phase 14', 'TEST access']
print('This notebook performs: frozen-artifact reading only.')
print('It does NOT perform:')
for item in forbidden:
    print('  -', item)
print()
print('next_action = WAIT_FOR_SUPERVISOR_PHASE13_PART1C_AUTHORIZATION')


This notebook performs: frozen-artifact reading only.
It does NOT perform:
  - standalone proposal analysis
  - predicted-region segmentation
  - regional integration gate
  - Phase 14
  - TEST access

next_action = WAIT_FOR_SUPERVISOR_PHASE13_PART1C_AUTHORIZATION
